In [1]:
import numpy as np
import pandas as pd
import fastf1 as f1

In [2]:
circuits = pd.read_csv('data/circuits.csv')
constructor_results = pd.read_csv('data/constructor_results.csv')
constructor_standings = pd.read_csv('data/constructor_standings.csv')
constructors = pd.read_csv('data/constructors.csv')
driver_standings = pd.read_csv('data/driver_standings.csv')
drivers = pd.read_csv('data/drivers.csv')
lap_times = pd.read_csv('data/lap_times.csv')
pit_stops = pd.read_csv('data/pit_stops.csv')
qualifying = pd.read_csv('data/qualifying.csv')
races = pd.read_csv('data/races.csv')
results = pd.read_csv('data/results.csv')
sprint_results = pd.read_csv('data/sprint_results.csv')
status = pd.read_csv('data/status.csv')
hungary = f1.get_session(2024, 13, "Race")
belgium = f1.get_session(2024, 14, "Race")

req         WARNING 	DEFAULT CACHE ENABLED! (24.0 KB) /home/vscode/.cache/fastf1


In [3]:
hungary.load()
belgium.load()

core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.4.0]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

In [4]:
final = pd.merge(results, races, on="raceId")
final = pd.merge(final, circuits, on="circuitId")
final = pd.merge(final, drivers, on="driverId")

In [5]:
final.drop(columns=["resultId", "number_x", "positionText", "positionOrder", "points", "laps", "time_x", "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed", "statusId", "round", "name_x", "url_x", "fp1_date", "fp1_time", "fp2_date", "fp2_time", "fp3_date", "fp3_time", "quali_date", "quali_time", "sprint_date", "sprint_time", "name_y", "location", "country", "lat", "lng", "url_y", "number_y", "code", "forename", "surname", "dob", "nationality", "url", "alt"], inplace=True)

final = pd.merge(final, constructors, on="constructorId")

In [6]:
final.drop(columns=["name", "nationality", "url"], inplace=True)

In [7]:
final.rename(columns={"time_y":"time"}, inplace=True)

In [8]:
final.drop(final[final.year < 2010].index)

,raceId,driverId,constructorId,grid,position,year,circuitId,date,time,circuitRef,driverRef,constructorRef
20320,337,4,6,3,1,2010,3,2010-03-14,12:00:00,bahrain,alonso,ferrari
20321,337,13,6,2,2,2010,3,2010-03-14,12:00:00,bahrain,massa,ferrari
20322,337,1,1,4,3,2010,3,2010-03-14,12:00:00,bahrain,hamilton,mclaren
20323,337,20,9,1,4,2010,3,2010-03-14,12:00:00,bahrain,vettel,red_bull
20324,337,3,131,5,5,2010,3,2010-03-14,12:00:00,bahrain,rosberg,mercedes
...,...,...,...,...,...,...,...,...,...,...,...,...
26514,1132,839,214,18,16,2024,9,2024-07-07,14:00:00,silverstone,ocon,alpine
26515,1132,815,9,0,17,2024,9,2024-07-07,14:00:00,silverstone,perez,red_bull
26516,1132,855,15,14,18,2024,9,2024-07-07,14:00:00,silverstone,zhou,sauber
26517,1132,847,131,1,\N,2024,9,2024-07-07,14:00:00,silverstone,russell,mercedes


In [9]:
races = [belgium, hungary]

In [10]:
for race in races:
    race_results = race.results
    race_results.drop(columns=["DriverNumber", "BroadcastName", "Abbreviation", "TeamColor", "FirstName", "LastName", "FullName", "HeadshotUrl", "CountryCode", "ClassifiedPosition", "Q1", "Q2", "Q3", "Time", "Status"], inplace=True)
    race_results = pd.merge(race_results, drivers, left_on="DriverId", right_on="driverRef")
    race_results["year"] = race.session_info["StartDate"].date().year
    race_results["date"] = race.session_info["StartDate"].date().strftime("%Y-%m-%d")
    race_results["time"] = race.session_info["StartDate"].time().strftime("%H:%M")
    if race.session_info["Meeting"]["Circuit"]["ShortName"] == "Spa-Francorchamps":
        # TODO come up with better way to add circuit names
        race_results["circuitRef"] = "spa"
        race_results["raceId"] = 1134
    else:
        race_results["circuitRef"] = "hungaroring"
        race_results["raceId"] = 1133
    race_results = pd.merge(race_results, circuits, on="circuitRef")
    race_results = pd.merge(race_results, constructors, left_on="TeamId", right_on="constructorRef")
    race_results.drop(columns=["DriverId", "TeamName", "TeamId", "Points", "number", "code", "country", "lat", "lng", "alt", "url_y", "name_y", "nationality_y", "url", "forename", "surname", "dob", "nationality_x", "url_x", "name_x", "location"], inplace=True)
    race_results.rename(columns={"Position":"position", "GridPosition":"grid"}, inplace=True)
    final = pd.concat([final, race_results], axis=0)

In [11]:
final.replace(to_replace="\\N", value=np.nan, inplace=True)

In [12]:
final.drop(columns=["driverId", "constructorId", "circuitId"], inplace=True)

In [13]:
final["circuit_code"] = final["circuitRef"].astype("category").cat.codes
final["driver_code"] = final["driverRef"].astype("category").cat.codes
final["constructor_code"] = final["constructorRef"].astype("category").cat.codes

In [14]:
final["position"] = final["position"].astype(float)

In [15]:
final["pos_delta"] = final["grid"] - final["position"]

In [16]:
final

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta
0,18,1.0,1.0,2008,2008-03-16,04:30:00,albert_park,hamilton,mclaren,3,343,131,0.0
1,18,5.0,2.0,2008,2008-03-16,04:30:00,albert_park,heidfeld,bmw_sauber,3,359,18,3.0
2,18,7.0,3.0,2008,2008-03-16,04:30:00,albert_park,rosberg,williams,3,686,208,4.0
3,18,11.0,4.0,2008,2008-03-16,04:30:00,albert_park,alonso,renault,3,20,166,7.0
4,18,3.0,5.0,2008,2008-03-16,04:30:00,albert_park,kovalainen,mclaren,3,436,131,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15,1133,12.0,16.0,2024,2024-07-21,15:00,hungaroring,bottas,sauber,25,102,168,-4.0
16,1133,14.0,17.0,2024,2024-07-21,15:00,hungaroring,sargeant,williams,25,705,208,-3.0
17,1133,19.0,18.0,2024,2024-07-21,15:00,hungaroring,ocon,alpine,25,583,5,1.0
18,1133,18.0,19.0,2024,2024-07-21,15:00,hungaroring,zhou,sauber,25,855,168,-1.0


In [17]:
def rolling_finish_avg(group, cols, new_cols):
    group = group.sort_values("raceId")
    rolling_stats = group[cols].rolling(3, closed="left").mean()
    group[new_cols] = rolling_stats
    return group

In [18]:
cols = ["grid", "position", "pos_delta"]
new_cols = [f"{c}_rolling" for c in cols]

In [19]:
final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")

/tmp/ipykernel_14362/1356228569.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rolling = final.groupby("driverRef").apply(lambda x: rolling_finish_avg(x, cols, new_cols)).droplevel("driverRef")


In [20]:
final_rolling = final_rolling.sort_values(["raceId", "position"])

In [21]:
final_rolling.index = range(final_rolling.shape[0])

In [22]:
final_rolling.to_csv("data/final_rolling.csv", index=False)

In [28]:
single = final_rolling.loc[final_rolling["raceId"] == 1134]

In [29]:
single

,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,circuit_code,driver_code,constructor_code,pos_delta,grid_rolling,position_rolling,pos_delta_rolling
26539,1134,3.0,1.0,2024,2024-07-28,15:00,spa,hamilton,mercedes,65,343,136,2.0,4.000000,2.666667,1.333333
26540,1134,5.0,2.0,2024,2024-07-28,15:00,spa,piastri,mclaren,65,617,131,3.0,4.666667,2.333333,2.333333
26541,1134,1.0,3.0,2024,2024-07-28,15:00,spa,leclerc,ferrari,65,461,69,-2.0,7.666667,9.666667,-2.000000
26542,1134,11.0,4.0,2024,2024-07-28,15:00,spa,max_verstappen,red_bull,65,519,165,7.0,2.666667,4.000000,-1.333333
26543,1134,4.0,5.0,2024,2024-07-28,15:00,spa,norris,mclaren,65,580,131,-1.0,2.000000,8.333333,-6.333333
26544,1134,7.0,6.0,2024,2024-07-28,15:00,spa,sainz,ferrari,65,699,69,1.0,5.000000,4.666667,0.333333
26545,1134,2.0,7.0,2024,2024-07-28,15:00,spa,perez,red_bull,65,607,165,-5.0,8.000000,10.333333,-2.333333
26546,1134,8.0,8.0,2024,2024-07-28,15:00,spa,alonso,aston_martin,65,20,11,0.0,10.666667,12.333333,-1.666667
26547,1134,9.0,9.0,2024,2024-07-28,15:00,spa,ocon,alpine,65,583,5,0.0,15.666667,15.333333,0.333333
26548,1134,13.0,10.0,2024,2024-07-28,15:00,spa,ricciardo,rb,65,670,162,3.0,11.666667,11.333333,0.333333
